In [1]:
import matplotlib
matplotlib.rcParams.update({'font.size' : 22})
import matplotlib.pyplot as plt
import pandas as pd
pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_colwidth', None)
import numpy as np
import joblib

In [3]:
import xgboost as xgb

In [4]:
'''
The "xgb_to_power" model predicts a power. 
Therefore, only the implant power prediction error can initially be calculated. 
It is possible to convert the power prediction error into spherical equivalent prediction error 
by multiplying this value by a factor of 0.7; however, it is more accurate to use a thin lens formula 
to account for variations in factors for different axial lengths and powers. 
The spherical equivalent associated with the predicted power can be determined 
by first calculating the thin lens ELP that results in the target (actual spherical equivalent) 
given the power predicted by the formula; then using this ELP value with the actual implanted power 
to determine the corresponding refraction.
'''

def solve_d_Haigis(AL, R, power, Rx): 
    '''
    This function calculates the "d" value (thin lens ELP) of the Haigis formula
    '''
    # input and output values in mm
    AL = AL/1000
    R = R/1000
    n = 1.336
    nc = 1.3315 
    dx = 0.012 
    Dc = (nc - 1) / R
    z = Dc + Rx/(1-Rx*dx) 
    b = -power * (AL + n/z) 
    c = (n * (AL - n/z) + power * AL * n/z) 
    d = (1 / (2*power)) * (-b - np.sqrt(b**2 - 4 * power * c) ) * 1000
    return d

def whichES(LA, R, d, power):
    '''
    This function determines the predicted spherical equivalent, for a given power and d value
    '''
    n = 1.336
    nc = 1.3315
    dx = 0.012
    Dc = (nc - 1)/(R*0.001)
    qnum = n * (n - power * ((LA/1000) - (d/1000)))
    qdenum = n * ((LA/1000) - (d/1000)) + (d/1000) * (n - power *((LA/1000)-(d/1000)))
    q = qnum / qdenum
    target = round((q - Dc) / (1 + dx*(q - Dc)),3)
    return target

In [5]:
# Specify input and output paths
output_path = '-- precise output path, where the model are saved --'
input_path = '-- precise input_path, where the datasets are saved --'

In [7]:
df = pd.read_excel('/Volumes/Bases/output/Eval_Article_3sets1403/' + 'Set2_withthin.xlsx')

In [8]:
df['K'] = np.sqrt(df['K1']*df['K2'])
df['R'] = 337.5/df['K']

In [9]:
# load models
AItoSE = xgb.Booster()
AItoSE.load_model(output_path + "AItoSE.model")

AItoPower = xgb.Booster()
AItoPower.load_model(output_path + "AItoPower.model")

In [10]:
# reprecise feature column names
AItoPower_lsfeat = ['AL', 'K', 'AQD', 'LT', 'CCT', 'WTW', 'ES_postop'] 
AItoSE_lsfeat = ['AL', 'K', 'AQD', 'LT', 'CCT', 'WTW', 'Power'    ]

In [11]:
# DMatrix creation
AItoPower_dtest = xgb.DMatrix(df[AItoPower_lsfeat].values)
AItoSE_dtest = xgb.DMatrix(df[AItoSE_lsfeat].values)

# predictions
AItoPower_pred = AItoPower.predict(AItoPower_dtest)
AItoSE_pred = AItoSE.predict(AItoSE_dtest)

# prediction columns creation
df['AItoPower'] = AItoPower_pred
df['AItoSE'] = AItoSE_pred

# conversion of predicted power to predicted SE for the xgb_to_power model
df['AItoPower_d'] = df[['AL', 'R', 'AItoPower', 'ES_postop']].apply(lambda x : solve_d_Haigis(*x), axis = 1)
df['AItoPower_SE'] = df[['AL', 'R', 'AItoPower_d', 'Power']].apply(lambda x : whichES(*x), axis = 1)

In [12]:
# prediction error calculation
df['AItoPower_SE_error'] = df['ES_postop'] - df['AItoPower_SE']
df['AItoSE_error']    = df['ES_postop'] - df['AItoSE']

In [13]:
# mean error for xgb_to_SE :
df['AItoSE_error'].mean()

0.32666821796704754

In [14]:
# mean error for xgb_to_power :
df['AItoPower_SE_error'].mean()

-0.037549618320610684

In [15]:
# formula adjustement : 
df['AItoSE'] += df['AItoSE_error'].mean()
df['AItoPower_SE'] += df['AItoPower_SE_error'].mean()

In [ ]:
# prediction errors are calculated and formulas are ready to be evaluated.